In [5]:
# mnist_unlearn_cumulative_1_3_5_v3.py
# 累积连续遗忘（严格版本）
# 修改说明：
# - 从根目录加载预训练模型和分类器
# - 所有本次实验产物（权重、图片）均保存到根目录下的 naive 文件夹
# - 降低了遗忘强度（epoch=3, lr=5e-5）

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torchdiffeq
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torchvision.transforms import ToPILImage
from PIL import Image, ImageDraw, ImageFont

from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchcfm.models.unet import UNetModel

# ---------------- core config ----------------
FORGET_SEQUENCE = [1, 3, 5]
SEED = 0
USE_TORCH_DIFFEQ = True
# --------------------------------------------

def set_seed(seed: int):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 轻量 MNIST 分类器
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*7*7, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def build_or_load_classifier(savedir, device, train_loader, test_loader, epochs=2, lr=1e-3):
    path = os.path.join(savedir, "mnist_cnn_classifier.pt")
    model = MNISTClassifier().to(device)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        print(f"[Classifier] 已加载分类器: {path}")
        return model

    print("[Classifier] 未找到分类器权重，开始训练...")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    model.train()
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward(); opt.step()
        # 验证
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(1)
                correct += (pred == y).sum().item()
                total += y.numel()
        acc = 100.0 * correct / total
        print(f"[Classifier] epoch {ep+1}/{epochs} acc {acc:.2f}%")
        model.train()
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"[Classifier] 分类器训练完成并保存: {path}")
    return model

def generate_grid(model, z0, generated_class_list, device):
    with torch.no_grad():
        if USE_TORCH_DIFFEQ:
            traj = torchdiffeq.odeint(
                lambda t, x: model.forward(t, x, generated_class_list),
                z0, torch.linspace(0, 1, 2, device=device),
                atol=1e-4, rtol=1e-4, method="dopri5",
            )
            x_final = traj[-1]
        else:
            x_final = z0
        grid = make_grid(x_final.view([-1,1,28,28]).clip(-1,1),
                         value_range=(-1,1), padding=2, nrow=10)
    return grid

@torch.no_grad()
def comprehensive_evaluation(model, clf, device, target_classes, n_samples=4096, n_classes=10):
    model.eval()
    y_rand = torch.randint(0, n_classes, (n_samples,), device=device)
    z = torch.randn(n_samples, 1, 28, 28, device=device)
    traj = torchdiffeq.odeint(
        lambda t, x: model.forward(t, x, y_rand),
        z, torch.linspace(0, 1, 2, device=device),
        atol=1e-4, rtol=1e-4, method="dopri5",
    )
    xg = traj[-1].clamp(-1, 1)
    logits = clf(xg)
    preds = logits.argmax(1).cpu().numpy()
    rates = {}
    tot = preds.shape[0]
    for cls in target_classes:
        rates[cls] = float((preds == cls).sum() / tot * 100.0)
    return rates

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # ---------- 路径修改 ----------
    # 根目录，用于加载预训练模型和分类器
    root_dir = "/mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist"
    # 输出目录，用于保存本次实验的所有产物
    output_dir = os.path.join(root_dir, "naive")
    os.makedirs(output_dir, exist_ok=True)
    
    # 从根目录加载预训练权重
    pretrain_path = os.path.join(root_dir, "cfm_mnist_pretrain.pt")
    # 对比图片保存到 naive 文件夹
    compare_img = os.path.join(output_dir, "mnist_cumulative_compare.png")

    if not os.path.exists(pretrain_path):
        raise FileNotFoundError(f"未找到预训练权重: {pretrain_path}")

    teacher = torch.load(pretrain_path, map_location=device).eval()
    print(f"已加载预训练模型: {pretrain_path}")

    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = datasets.MNIST("../data", train=True, download=True, transform=tf)
    testset  = datasets.MNIST("../data", train=False, download=True, transform=tf)
    train_loader_cls = DataLoader(trainset, batch_size=256, shuffle=True)
    test_loader_cls  = DataLoader(testset, batch_size=512, shuffle=False)
    # 从根目录加载或构建分类器
    classifier = build_or_load_classifier(root_dir, device, train_loader_cls, test_loader_cls, epochs=2)

    # 计算基线生成率
    prev_rates = comprehensive_evaluation(teacher, classifier, device, target_classes=FORGET_SEQUENCE, n_samples=4096)
    print("[Baseline] 预训练模型生成率:")
    for cls in FORGET_SEQUENCE:
        print(f"  类 {cls}: {prev_rates[cls]:.4f}%")

    # 初始化 student 为 teacher（累积遗忘）
    student = torch.load(pretrain_path, map_location=device)
    student.train()
    batch_size = 128
    # ---------- 降低遗忘强度：减少训练轮数和学习率 ----------
    n_epochs_per_stage = 3
    lr = 5e-5
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
    FM = ConditionalFlowMatcher(sigma=0.0)

    # z0 用于可视化一致性
    z0 = torch.randn(100, 1, 28, 28, device=device)
    generated_class_list = torch.arange(10, device=device).repeat(10)

    grids = {}
    grids['pretrain'] = generate_grid(teacher, z0.clone(), generated_class_list, device)

    t_start = time.time()
    
    forgotten_classes = []

    # ---------- 累积遗忘阶段 ----------
    for stage, forget_class in enumerate(FORGET_SEQUENCE):
        forgotten_classes.append(forget_class)
        print(f"\n=== 阶段 {stage+1}/{len(FORGET_SEQUENCE)}: 遗忘类别 {forget_class} (累积遗忘: {forgotten_classes}) ===")
        student.train()
        optimizer = torch.optim.Adam(student.parameters(), lr=lr)

        for epoch in range(n_epochs_per_stage):
            for x_real, y in train_loader:
                x_real, y = x_real.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                x_noise = torch.randn_like(x_real)
                x_target = x_real.clone()
                forgotten_tensor = torch.tensor(forgotten_classes, device=device)
                mask_forget = torch.isin(y, forgotten_tensor)
                if mask_forget.any():
                    x_target[mask_forget] = x_noise[mask_forget]
                t, xt, ut = FM.sample_location_and_conditional_flow(x_noise, x_target)
                vt = student(t, xt, y)
                loss = torch.mean((vt - ut)**2)
                loss.backward()
                optimizer.step()

        # 阶段结束后将权重保存到 output_dir (naive文件夹)
        stage_weight_path = os.path.join(output_dir, f"cfm_mnist_unlearn_stage{stage+1}.pt")
        torch.save(student.state_dict(), stage_weight_path)
        student.eval()

        # 输出各类生成率
        stage_rates = comprehensive_evaluation(student, classifier, device, target_classes=FORGET_SEQUENCE, n_samples=4096)
        print(f"阶段 {stage+1} 生成率:")
        for cls in FORGET_SEQUENCE:
            abs_drop = prev_rates[cls] - stage_rates[cls]
            print(f"  类 {cls}: {stage_rates[cls]:.4f}% | 相比上阶段下降 {abs_drop:.4f}%")

        prev_rates = stage_rates

        # 可视化网格
        key = f"after_stage{stage+1}"
        grids[key] = generate_grid(student, z0.clone(), generated_class_list, device)

    t_total = time.time() - t_start

    # 保存最终权重到 output_dir (naive文件夹)
    final_weight_path = os.path.join(output_dir, "cfm_mnist_unlearn_final.pt")
    torch.save(student.state_dict(), final_weight_path)
    print(f"\n最终累积遗忘权重已保存: {final_weight_path}")
    print(f"总耗时: {t_total:.1f}s")

    # ---------- 四阶段网格图拼接 ----------
    pil_grids = [ToPILImage()(grids[k].cpu()) for k in ['pretrain','after_stage1','after_stage2','after_stage3']]
    widths, heights = zip(*(img.size for img in pil_grids))
    total_width = sum(widths) + 30*3  # 每张图间距 30
    max_height = max(heights)
    canvas = Image.new('RGB', (total_width, max_height + 50), color=(255,255,255))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except IOError:
        font = ImageFont.load_default()

    x_offset = 0
    titles = ["Pretrain","After Stage 1","After Stage 2","After Stage 3"]
    for i, img in enumerate(pil_grids):
        canvas.paste(img, (x_offset,50))
        draw.text((x_offset,10), titles[i], fill=(0,0,0), font=font)
        x_offset += img.size[0] + 30
    canvas.save(compare_img)
    print(f"四阶段对比图已保存: {compare_img}")

if __name__ == "__main__":
    main()

使用设备: cuda
已加载预训练模型: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/cfm_mnist_pretrain.pt
[Classifier] 已加载分类器: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/mnist_cnn_classifier.pt
[Baseline] 预训练模型生成率:
  类 1: 10.0342%
  类 3: 10.0342%
  类 5: 10.5225%

=== 阶段 1/3: 遗忘类别 1 (累积遗忘: [1]) ===
阶段 1 生成率:
  类 1: 0.0732% | 相比上阶段下降 9.9609%
  类 3: 10.7422% | 相比上阶段下降 -0.7080%
  类 5: 11.6699% | 相比上阶段下降 -1.1475%

=== 阶段 2/3: 遗忘类别 3 (累积遗忘: [1, 3]) ===
阶段 2 生成率:
  类 1: 0.2197% | 相比上阶段下降 -0.1465%
  类 3: 1.5869% | 相比上阶段下降 9.1553%
  类 5: 12.5977% | 相比上阶段下降 -0.9277%

=== 阶段 3/3: 遗忘类别 5 (累积遗忘: [1, 3, 5]) ===
阶段 3 生成率:
  类 1: 0.1221% | 相比上阶段下降 0.0977%
  类 3: 2.2217% | 相比上阶段下降 -0.6348%
  类 5: 3.7842% | 相比上阶段下降 8.8135%

最终累积遗忘权重已保存: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/naive/cfm_mnist_unlearn_final.pt
总耗时: 384.8s
四阶段对比图已保存: /mnt/afs/int

In [2]:
# mnist_unlearn_cumulative_1_3_5_orthogonality_v6.py
# 累积连续遗忘（最终版：正交性约束 + 选择性蒸馏）
# 修改说明：
# - 核心修改：引入全新的“选择性蒸馏”机制(L_distill)，强制模型在处理“保留类”数据时
#   的行为与上一阶段保持一致，极大地稳定了模型，从根本上防止参数漂移导致的遗忘衰退。
# - 综合策略：将强力的正交性约束(L_ortho)与选择性蒸馏(L_distill)相结合，
#   形成“防守”+“稳固”的双重巩固机制。
# - 参数设置：引入新超参数GAMMA控制蒸馏强度，并保持其他有效设置。
# - 所有实验产物保存到 orthogonality_v6 文件夹。

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torchdiffeq
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torchvision.transforms import ToPILImage
from PIL import Image, ImageDraw, ImageFont

from torch.optim.lr_scheduler import CosineAnnealingLR

from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchcfm.models.unet import UNetModel

# ---------------- core config ----------------
FORGET_SEQUENCE = [1, 3, 5]
SEED = 0
USE_TORCH_DIFFEQ = True
# ---------- Regularization Config (双重巩固) ----------
# 1. 正交性约束强度 (防守历史遗忘类)
BETA = 50.0
# 2. 新增：选择性蒸馏强度 (稳固保留类)
GAMMA = 5.0
# -----------------------------------------------------------------------

def set_seed(seed: int):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# 轻量 MNIST 分类器 (保持不变)
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*7*7, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def build_or_load_classifier(savedir, device, train_loader, test_loader, epochs=2, lr=1e-3):
    path = os.path.join(savedir, "mnist_cnn_classifier.pt")
    model = MNISTClassifier().to(device)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        print(f"[Classifier] 已加载分类器: {path}")
        return model

    print("[Classifier] 未找到分类器权重，开始训练...")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    model.train()
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward(); opt.step()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x).argmax(1)
                correct += (pred == y).sum().item()
                total += y.numel()
        acc = 100.0 * correct / total
        print(f"[Classifier] epoch {ep+1}/{epochs} acc {acc:.2f}%")
        model.train()
    model.eval()
    torch.save(model.state_dict(), path)
    print(f"[Classifier] 分类器训练完成并保存: {path}")
    return model

def generate_grid(model, z0, generated_class_list, device):
    with torch.no_grad():
        if USE_TORCH_DIFFEQ:
            traj = torchdiffeq.odeint(
                lambda t, x: model.forward(t, x, generated_class_list),
                z0, torch.linspace(0, 1, 2, device=device),
                atol=1e-4, rtol=1e-4, method="dopri5",
            )
            x_final = traj[-1]
        else:
            x_final = z0
        grid = make_grid(x_final.view([-1,1,28,28]).clip(-1,1),
                         value_range=(-1,1), padding=2, nrow=10)
    return grid

@torch.no_grad()
def comprehensive_evaluation(model, clf, device, target_classes, n_samples=4096, n_classes=10):
    model.eval()
    y_rand = torch.randint(0, n_classes, (n_samples,), device=device)
    z = torch.randn(n_samples, 1, 28, 28, device=device)
    traj = torchdiffeq.odeint(
        lambda t, x: model.forward(t, x, y_rand),
        z, torch.linspace(0, 1, 2, device=device),
        atol=1e-4, rtol=1e-4, method="dopri5",
    )
    xg = traj[-1].clamp(-1, 1)
    logits = clf(xg)
    preds = logits.argmax(1).cpu().numpy()
    rates = {}
    tot = preds.shape[0]
    for cls in target_classes:
        rates[cls] = float((preds == cls).sum() / tot * 100.0)
    return rates

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # ---------- 路径修改 ----------
    root_dir = "/mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist"
    output_dir = os.path.join(root_dir, "orthogonality_v6") # <--- 修改点：新的输出文件夹
    os.makedirs(output_dir, exist_ok=True)
    
    pretrain_path = os.path.join(root_dir, "cfm_mnist_pretrain.pt")
    compare_img = os.path.join(output_dir, "mnist_cumulative_compare_ortho_v6.png") # <--- 修改点：新的图片文件名

    if not os.path.exists(pretrain_path):
        raise FileNotFoundError(f"未找到预训练权重: {pretrain_path}")

    teacher = torch.load(pretrain_path, map_location=device).eval()
    print(f"已加载预训练模型: {pretrain_path}")

    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = datasets.MNIST("../data", train=True, download=True, transform=tf)
    testset  = datasets.MNIST("../data", train=False, download=True, transform=tf)
    train_loader_cls = DataLoader(trainset, batch_size=256, shuffle=True)
    test_loader_cls  = DataLoader(testset, batch_size=512, shuffle=False)
    classifier = build_or_load_classifier(root_dir, device, train_loader_cls, test_loader_cls, epochs=2)

    prev_rates = comprehensive_evaluation(teacher, classifier, device, target_classes=FORGET_SEQUENCE, n_samples=4096)
    print("[Baseline] 预训练模型生成率:")
    for cls in FORGET_SEQUENCE:
        print(f"  类 {cls}: {prev_rates[cls]:.4f}%")

    student = torch.load(pretrain_path, map_location=device)
    student.train()
    batch_size = 128
    n_epochs_per_stage = 8
    lr = 5e-5
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
    FM = ConditionalFlowMatcher(sigma=0.0)

    z0 = torch.randn(100, 1, 28, 28, device=device)
    generated_class_list = torch.arange(10, device=device).repeat(10)

    grids = {}
    grids['pretrain'] = generate_grid(teacher, z0.clone(), generated_class_list, device)

    t_start = time.time()
    
    forgotten_classes = []
    historical_forgotten_classes = [] 

    # ---------- 累积遗忘阶段 ----------
    for stage, forget_class in enumerate(FORGET_SEQUENCE):
        student_prev = None
        if historical_forgotten_classes:
            print("[Consolidation] 创建上一阶段模型的冻结副本...")
            student_prev = copy.deepcopy(student).eval()
            for p in student_prev.parameters():
                p.requires_grad = False

        forgotten_classes.append(forget_class)
        print(f"\n=== 阶段 {stage+1}/{len(FORGET_SEQUENCE)}: 遗忘类别 {forget_class} (累积遗忘: {forgotten_classes}) ===")
        student.train()
        
        optimizer = torch.optim.Adam(student.parameters(), lr=lr)
        scheduler = CosineAnnealingLR(optimizer, T_max=len(train_loader) * n_epochs_per_stage)

        for epoch in range(n_epochs_per_stage):
            for x_real, y in train_loader:
                x_real, y = x_real.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                x_noise = torch.randn_like(x_real)
                
                # --- 1. 计算标准的遗忘损失 (L_unlearn) ---
                x_target = x_real.clone()
                forgotten_tensor = torch.tensor(forgotten_classes, device=device)
                mask_forget = torch.isin(y, forgotten_tensor)
                if mask_forget.any():
                    x_target[mask_forget] = x_noise[mask_forget]
                
                t, xt, ut = FM.sample_location_and_conditional_flow(x_noise, x_target)
                vt = student(t, xt, y)
                loss_unlearn = torch.mean((vt - ut)**2)

                loss_ortho = torch.tensor(0.0, device=device)
                loss_distill = torch.tensor(0.0, device=device)

                # 只有在有历史遗忘（即进入第二阶段后）才需要巩固
                if student_prev:
                    # --- 2. 计算正交性损失 (L_ortho) ---
                    if historical_forgotten_classes:
                        hist_forgotten_tensor = torch.tensor(historical_forgotten_classes, device=device)
                        mask_hist_forget = torch.isin(y, hist_forgotten_tensor)
                        if mask_hist_forget.any():
                            xt_hist = xt[mask_hist_forget]
                            y_hist = y[mask_hist_forget]
                            t_hist = t[mask_hist_forget]
                            vt_current_hist = student(t_hist, xt_hist, y_hist)
                            with torch.no_grad():
                                vt_prev_hist = student_prev(t_hist, xt_hist, y_hist)
                            delta_vt = vt_current_hist - vt_prev_hist
                            inner_product = torch.sum(delta_vt.flatten(1) * vt_prev_hist.flatten(1), dim=1)
                            loss_ortho = torch.mean(torch.relu(inner_product))

                    # --- 3. 计算选择性蒸馏损失 (L_distill) ---
                    mask_retain = ~mask_forget
                    if mask_retain.any():
                        xt_retain = xt[mask_retain]
                        y_retain = y[mask_retain]
                        t_retain = t[mask_retain]
                        
                        vt_current_retain = student(t_retain, xt_retain, y_retain)
                        with torch.no_grad():
                            vt_prev_retain = student_prev(t_retain, xt_retain, y_retain)
                        loss_distill = torch.mean((vt_current_retain - vt_prev_retain)**2)

                # --- 4. 组合总损失并反向传播 ---
                total_loss = loss_unlearn + BETA * loss_ortho + GAMMA * loss_distill
                total_loss.backward()
                optimizer.step()
                scheduler.step()

        historical_forgotten_classes = list(forgotten_classes)

        stage_weight_path = os.path.join(output_dir, f"cfm_mnist_unlearn_stage{stage+1}_ortho_v6.pt")
        torch.save(student.state_dict(), stage_weight_path)
        student.eval()

        stage_rates = comprehensive_evaluation(student, classifier, device, target_classes=FORGET_SEQUENCE, n_samples=4096)
        print(f"阶段 {stage+1} 生成率:")
        for cls in FORGET_SEQUENCE:
            prev_rate = prev_rates.get(cls, 0.0) 
            abs_drop = prev_rate - stage_rates[cls]
            print(f"  类 {cls}: {stage_rates[cls]:.4f}% | 相比上阶段下降 {abs_drop:.4f}%")
        prev_rates = stage_rates

        key = f"after_stage{stage+1}"
        grids[key] = generate_grid(student, z0.clone(), generated_class_list, device)

    t_total = time.time() - t_start

    final_weight_path = os.path.join(output_dir, "cfm_mnist_unlearn_final_ortho_v6.pt")
    torch.save(student.state_dict(), final_weight_path)
    print(f"\n最终累积遗忘权重已保存: {final_weight_path}")
    print(f"总耗时: {t_total:.1f}s")

    # ---------- 四阶段网格图拼接 (保持不变) ----------
    pil_grids = [ToPILImage()(grids[k].cpu()) for k in ['pretrain','after_stage1','after_stage2','after_stage3']]
    widths, heights = zip(*(img.size for img in pil_grids))
    total_width = sum(widths) + 30*3
    max_height = max(heights)
    canvas = Image.new('RGB', (total_width, max_height + 50), color=(255,255,255))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("arial.ttf", 20)
    except IOError:
        font = ImageFont.load_default()

    x_offset = 0
    titles = ["Pretrain","After Stage 1 (Forget 1)","After Stage 2 (Forget 1,3)","After Stage 3 (Forget 1,3,5)"]
    for i, img in enumerate(pil_grids):
        canvas.paste(img, (x_offset,50))
        draw.text((x_offset,10), titles[i], fill=(0,0,0), font=font)
        x_offset += img.size[0] + 30
    canvas.save(compare_img)
    print(f"四阶段对比图已保存: {compare_img}")

if __name__ == "__main__":
    main()

使用设备: cuda
已加载预训练模型: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/cfm_mnist_pretrain.pt
[Classifier] 已加载分类器: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/mnist_cnn_classifier.pt
[Baseline] 预训练模型生成率:
  类 1: 10.0342%
  类 3: 10.0342%
  类 5: 10.5225%

=== 阶段 1/3: 遗忘类别 1 (累积遗忘: [1]) ===
阶段 1 生成率:
  类 1: 0.1221% | 相比上阶段下降 9.9121%
  类 3: 10.3516% | 相比上阶段下降 -0.3174%
  类 5: 11.6943% | 相比上阶段下降 -1.1719%
[Consolidation] 创建上一阶段模型的冻结副本...

=== 阶段 2/3: 遗忘类别 3 (累积遗忘: [1, 3]) ===
阶段 2 生成率:
  类 1: 0.0488% | 相比上阶段下降 0.0732%
  类 3: 1.7334% | 相比上阶段下降 8.6182%
  类 5: 12.5732% | 相比上阶段下降 -0.8789%
[Consolidation] 创建上一阶段模型的冻结副本...

=== 阶段 3/3: 遗忘类别 5 (累积遗忘: [1, 3, 5]) ===
阶段 3 生成率:
  类 1: 0.0732% | 相比上阶段下降 -0.0244%
  类 3: 1.9775% | 相比上阶段下降 -0.2441%
  类 5: 3.4668% | 相比上阶段下降 9.1064%

最终累积遗忘权重已保存: /mnt/afs/intern/fangwenhan/wzz-y/sss/conditional-flow-matching/examples/2D_tutorials/models/cond_mnist/or